# Report: cohort kept by a round-2 gate around each 1000G population
Reads the existing `round2_pca` scores; writes only its figures and masked tables to `report/pop_gates` of `RUN_ID`. One run per superpopulation (the first in `RUN_IDS` with a PCA). Same gate as `ancestry_round2`: MCD on the population's founders, first `r2_k` PCs, radius from `r2_q`. Gates overlap, so rows don't add up to `n_round1`. Counts 1-20 are masked.

In [ ]:
RUN_ID = "chb_chs"   # only sets up the session

In [ ]:
RUN_IDS = ["chb_chs", "yri", "lwk", "stu_itu_beb", "asw", "mxl"]
K = None   # None: each run's r2_k
Q = None   # None: each run's r2_q

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_IDS[0])
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
def mahalanobis(Z, mu, C):
    d = Z[:, :len(mu)] - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, np.linalg.inv(C), d))

def gate_ellipse(ax, mu, C, i, j, r, **kw):
    """Outline of the ellipsoid on PCs i, j."""
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(C[np.ix_([i, j], [i, j])])
    ax.add_patch(Ellipse((mu[i], mu[j]), 2 * r * np.sqrt(vals[-1]), 2 * r * np.sqrt(vals[0]),
                         angle=np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1])), fill=False, **kw))

def fit_robust_gate(A, q, seed=0):
    """Reweighted MCD on the target pops; radius = sqrt(chi2_K quantile q). MCD, not OGK:
    OGK's scatter isn't consistency-corrected, so chi-square radii on it are off."""
    from scipy.stats import chi2
    from sklearn.covariance import MinCovDet
    mcd = MinCovDet(random_state=seed).fit(A)
    mu, C = mcd.location_, mcd.covariance_
    return {"mu": mu, "C": C, "radius": float(np.sqrt(chi2.ppf(q, A.shape[1]))),
            "ref_d": mahalanobis(A, mu, C), "anchor_outliers": int((~mcd.support_).sum())}

def peak_enrichment(loadings, pcs, window_mb=10):
    """Max over windows of (share of squared loading / share of variants); ~1 = spread out."""
    w = loadings["chr"].astype(str) + ":" + (loadings["pos"] // (window_mb * 1e6)).astype(int).astype(str)
    share_var = w.value_counts(normalize=True)
    out = {}
    for pc in pcs:
        sq = loadings[pc] ** 2
        share = sq.groupby(w).sum() / sq.sum()
        out[pc] = float((share / share_var.reindex(share.index)).max())
    return pd.Series(out)

def hexbin_safe(ax, x, y, **kw):
    """Participant density; cells with fewer than 21 people are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

In [ ]:
rows, seen, inside = [], set(), {}
for run_id in RUN_IDS:
    c = load_run(run_id)
    sp, path = c.param("superpop"), single_target(c, "round2_pca")
    if sp in seen or not get_store(path).exists(path):
        print(run_id, sp, "same superpop as an earlier run" if sp in seen else "no round2_pca yet"); continue
    seen.add(sp)
    k, q = K or c.param("r2_k"), Q or c.param("r2_q")
    s = pd.read_csv(path, sep="\t", dtype={"id": str})
    ref, coh = s[s["set"] == "reference"], s[s["set"] == "cohort"]
    pcs = [p for p in s if p.startswith("PC")][:k]
    X = coh[pcs].to_numpy()
    for pop, d in ref.groupby("pop"):
        g = fit_robust_gate(d[pcs].to_numpy(), q)
        inside[(sp, pop)] = mahalanobis(X, g["mu"], g["C"]) <= g["radius"]
        rows.append({"superpop": sp, "pop": pop, "pca_run": run_id, "k": k, "q": q, "n_ref": len(d),
                     "ref_inside": round(float((g["ref_d"] <= g["radius"]).mean()), 3), "n_round1": len(coh),
                     "n_kept": int(inside[(sp, pop)].sum())})
tab = pd.DataFrame(rows)
small = tab["n_kept"].between(1, 20)
tab["pct_of_round1"] = (100 * tab["n_kept"] / tab["n_round1"]).round(2).mask(small)
tab["n_kept"] = tab["n_kept"].astype(object).mask(small, "<=20")
OUTDIR = os.path.expanduser(cfg.path("report", "pop_gates"))   # read by scripts/dev/ancestry_report_pdf.py
os.makedirs(OUTDIR, exist_ok=True)
FILES = [{"file": "kept_by_population.tsv", "kind": "table", "caption": "counts 1-20 masked"}]
tab.to_csv(f"{OUTDIR}/kept_by_population.tsv", sep="\t", index=False)
def save_fig(fig, name, caption=""):
    fig.savefig(f"{OUTDIR}/{name}.png", dpi=130, bbox_inches="tight")
    FILES.append({"file": f"{name}.png", "kind": "figure", "caption": caption})
tab

In [ ]:
# kept per population, one panel per superpopulation (bars under 21 are not drawn)
sps = tab["superpop"].unique()
fig, axes = plt.subplots(1, len(sps), figsize=(4.5 * len(sps), 4), squeeze=False)
for ax, sp in zip(axes[0], sps):
    d = tab[tab["superpop"] == sp]
    n = [x if isinstance(x, (int, np.integer)) else 0 for x in d["n_kept"]]
    ax.barh(d["pop"], n, color="#2a78d6")
    for y, (x, shown) in enumerate(zip(n, d["n_kept"])):
        ax.text(x, y, f" {shown:,}" if x else f" {shown}", va="center", fontsize=9)
    ax.set(title=f"{sp} (round 1: {d['n_round1'].iloc[0]:,})", xlabel="AoU people inside the gate")
    ax.set_xlim(0, max(n + [1]) * 1.25); ax.invert_yaxis()
plt.tight_layout(); save_fig(fig, "kept_by_population", "AoU people inside each 1000G population's gate; bars under 21 not drawn"); plt.show()

In [ ]:
# overlap: people inside both gates (diagonal = the gate alone); counts 1-20 masked
for sp in sps:
    pops = [p for s_, p in inside if s_ == sp]
    M = np.array([[int((inside[(sp, a)] & inside[(sp, b)]).sum()) for b in pops] for a in pops])
    only = [int((inside[(sp, a)] & ~np.any([inside[(sp, b)] for b in pops if b != a], axis=0)).sum()) for a in pops]
    show = pd.DataFrame(M, index=pops, columns=pops).astype(object).mask((M >= 1) & (M <= 20), "<=20")
    show["only_this_gate"] = [o if not 1 <= o <= 20 else "<=20" for o in only]
    J = M / np.maximum(np.add.outer(np.diag(M), np.diag(M)) - M, 1)   # Jaccard
    fig, ax = plt.subplots(figsize=(1 + 0.7 * len(pops), 0.8 + 0.6 * len(pops)))
    im = ax.imshow(J, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(pops)), pops, rotation=45); ax.set_yticks(range(len(pops)), pops)
    for a in range(len(pops)):
        for b in range(len(pops)):
            ax.text(b, a, f"{J[a, b]:.2f}", ha="center", va="center", fontsize=8, color="white" if J[a, b] > 0.6 else "black")
    ax.set_title(f"{sp}: overlap of gates (Jaccard)"); fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); save_fig(fig, f"overlap_{sp}", "share of people in either gate who are in both"); plt.show()
    show.rename_axis("pop").reset_index().to_csv(f"{OUTDIR}/overlap_{sp}.tsv", sep="\t", index=False)
    FILES.append({"file": f"overlap_{sp}.tsv", "kind": "table", "caption": "people in both gates; counts 1-20 masked"})
    display(show)
json.dump({"run_id": "pop_gates", "stage": "pop_gates", "notes": {"runs": RUN_IDS, "k": K or "each run's r2_k", "q": Q or "each run's r2_q"},
           "checks": [], "files": FILES}, open(f"{OUTDIR}/summary.json", "w"), indent=2, default=str)
print("saved ->", OUTDIR)

`ref_inside` well under `q`: the population's MCD fit is unstable at this K (small or spread out).